# Assignment 2: Research Agent (Web Search + LLM)

**Goal:** Build a simple research agent that:
1. Takes a user question
2. Refines it into a search query (using LLM)
3. Searches the web (DuckDuckGo)
4. Fetches content from top results
5. Synthesizes a final answer (using LLM)

**Tools:** OpenRouter API, DuckDuckGo Search, BeautifulSoup

## 1. Setup & Imports

In [29]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from ddgs import DDGS
from bs4 import BeautifulSoup

load_dotenv(os.path.join(os.getcwd(), '..', '.env'))

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

FREE_MODEL = "arcee-ai/trinity-large-preview:free"
print("Client initialized successfully")

Client initialized successfully


## 2. Define Tools

### Tool 1: Internet Search (DuckDuckGo)
Returns top 3 URLs for a given query.

In [30]:
def internet_search(query):
    """Search DuckDuckGo and return top 3 result URLs."""
    print(f"Searching for: {query}...")
    try:
        results = DDGS().text(query, region='en-us', max_results=3)
        urls = [r['href'] for r in results]
        print(f"   Found {len(urls)} results")
        return urls
    except Exception as e:
        print(f"Search error: {e}")
        return []

print("internet_search defined")

internet_search defined


### Tool 2: Fetch URL Content
Scrapes a webpage and returns clean text (max 4000 chars).

In [31]:
def fetch_url(url):
    """Fetch and clean text content from a URL."""
    print(f"Fetching content from: {url}")
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, timeout=10, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        for script_or_style in soup(["script", "style"]):
            script_or_style.decompose()
            
        text = soup.get_text(separator=' ')
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        clean_text = '\n'.join(chunk for chunk in chunks if chunk)
        
        preview = clean_text[:300]
        print(f"    Fetched {len(clean_text)} chars (showing first 300):\n   {preview}...")
        return clean_text[:4000]
    except Exception as e:
        return f"Failed to retrieve content: {e}"

print("fetch_url defined")

fetch_url defined


## 3. Agent Logic

The agent follows a 4-step pipeline:
1. **Refine Query** - LLM turns the user question into a better search query
2. **Search** - DuckDuckGo returns top 3 URLs
3. **Fetch** - Scrape content from each URL
4. **Synthesize** - LLM generates a final answer from the collected data

In [32]:
def run_research_agent(user_question):
    """Run the full research agent pipeline."""
    print(f"Starting Research Agent...")
    print(f"   User Question: {user_question}\n")

    # STEP 1: Refine the query
    print("Step 1: Refining query with LLM")
    refine_msg = client.chat.completions.create(
        model=FREE_MODEL,
        messages=[{
            "role": "user",
            "content": (
                f"Convert this question into a short search engine query (max 10 words, no quotes, no explanation):\n"
                f"{user_question}"
            )
        }]
    )
    search_query = refine_msg.choices[0].message.content.strip().strip('"')
    print(f"    Refined Query: {search_query}\n")

    # STEP 2: Execute Search
    print("Step 2: Searching the web")
    top_3_links = internet_search(search_query)
    
    if not top_3_links:
        return "Sorry, I couldn't find any search results."
    
    for i, link in enumerate(top_3_links, 1):
        print(f"   {i}. {link}")
    print()

    # STEP 3: Fetch Data
    print("Step 3: Fetching content from sources")
    combined_source_data = ""
    for i, link in enumerate(top_3_links, 1):
        content = fetch_url(link)
        combined_source_data += f"\n--- DATA FROM SOURCE {i} ({link}) ---\n{content}\n"
    print()

    # STEP 4: Final Synthesis
    print("Step 4: Synthesizing final answer with LLM")
    system_prompt = (
        "You are an expert Research Agent. "
        "Use the provided content from the top-3 ranking sites to answer the user's question. "
        "Ensure your answer is based ONLY on the provided web data. Be professional and concise."
    )

    final_answer = client.chat.completions.create(
        model=FREE_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"User Question: {user_question}\n\nWeb Data:\n{combined_source_data}"}
        ]
    )
    
    return final_answer.choices[0].message.content

print("run_research_agent defined")

run_research_agent defined


## 4. Run the Agent

In [33]:
user_query = "What are the benefits of artificial intelligence in healthcare?"
result = run_research_agent(user_query)

print("\n" + "=" * 70)
print("FINAL RESEARCH REPORT")
print("=" * 70)
print(result)

Starting Research Agent...
   User Question: What are the benefits of artificial intelligence in healthcare?

Step 1: Refining query with LLM


    Refined Query: Benefits of AI in healthcare

Step 2: Searching the web
Searching for: Benefits of AI in healthcare...
   Found 3 results
   1. https://health.clevelandclinic.org/ai-in-healthcare
   2. https://pmc.ncbi.nlm.nih.gov/articles/PMC11612599/
   3. https://www.healthline.com/health/ai-healthcare-benefits

Step 3: Fetching content from sources
Fetching content from: https://health.clevelandclinic.org/ai-in-healthcare
    Fetched 15520 chars (showing first 300):
   How AI Is Used in Healthcare Locations: Abu Dhabi | Canada | Florida | London | Nevada | Ohio | Health Essentials Health Library Find a Provider Make an Appointment News Careers Contact Us Post or Recipes debug info:
client: {"assets":{},"datasets":{},"live":{},"projects":{},"users":{},"observable":...
Fetching content from: https://pmc.ncbi.nlm.nih.gov/articles/PMC11612599/
    Fetched 17 chars (showing first 300):
   403 403 Forbidden...
Fetching content from: https://www.healthline.com/health/ai-healthcare-bene